# Mean-variance portfolio

Classic Markowitz: maximize $\mu^\top w - w^\top \Sigma w$, where $\mu$ is the vector of mean daily returns and $\Sigma$ is the daily return covariance. Closed-form solution — just linear algebra.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stonks import get_prices, to_returns

%matplotlib inline


## Parameters


In [ ]:
PERIOD = "2y"
INTERVAL = "1d"
TOP_N = 50
FIELD = "close"   # adjusted close -> total returns


## Fetch prices and compute returns


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval=INTERVAL, field=FIELD)
returns = to_returns(prices).dropna()   # N tickers x (T-1) daily simple returns
print("prices:", prices.shape, "| returns:", returns.shape)


## Expected returns $\mu$ and covariance $\Sigma$

$\mu$ = mean daily return per stock; $\Sigma$ = N×N covariance of daily returns (built explicitly, since `returns.cov()` would treat dates as variables and return a dates×dates matrix).


In [ ]:
X = returns.to_numpy(dtype=float)
tickers = list(returns.index)
mu = X.mean(axis=1)                       # expected daily return per stock (N,)
Xc = X - mu[:, None]
Sigma = (Xc @ Xc.T) / (Xc.shape[1] - 1)   # N x N sample covariance (ddof=1)

print("mu shape:", mu.shape, "| Sigma shape:", Sigma.shape)
print("Sigma condition number: %.1e  (invertible while T > N)" % np.linalg.cond(Sigma))


## The optimization

Write the objective as $\max_w\ \mu^\top w - \tfrac{\gamma}{2}\, w^\top \Sigma w$; your formula is $\gamma = 2$. Setting $\nabla = 0$ gives the unconstrained optimum
$$w^* = \tfrac{1}{\gamma}\, \Sigma^{-1}\mu.$$


In [ ]:
gamma = 2.0   # risk aversion (your objective mu^T w - w^T Sigma w is gamma = 2)
w_unc = np.linalg.solve(Sigma, mu) / gamma
print("unconstrained: sum(w)=%.3f  min=%.3f  max=%.3f" % (w_unc.sum(), w_unc.min(), w_unc.max()))


### Fully invested ($\mathbf{1}^\top w = 1$)

Adding the budget constraint $\sum_i w_i = 1$, the closed form is
$$w^* = \tfrac{1}{\gamma}\bigl(\Sigma^{-1}\mu - \lambda\, \Sigma^{-1}\mathbf{1}\bigr),\quad \lambda = \frac{B - \gamma}{A},\quad A = \mathbf{1}^\top\Sigma^{-1}\mathbf{1},\quad B = \mathbf{1}^\top\Sigma^{-1}\mu.$$


In [ ]:
ones = np.ones(len(mu))
Sinv_mu = np.linalg.solve(Sigma, mu)
Sinv_1  = np.linalg.solve(Sigma, ones)
A = ones @ Sinv_1
B = ones @ Sinv_mu
lam = (B - gamma) / A
w = (Sinv_mu - lam * Sinv_1) / gamma   # sums to 1

print("sum(w)=%.6f  min=%.3f  max=%.3f  # shorts=%d" % (w.sum(), w.min(), w.max(), (w < 0).sum()))


## Result

Portfolio expected return and risk (annualized: ×252 for return, ×√252 for vol).


In [ ]:
ann_ret = w @ mu * 252
ann_vol = np.sqrt(w @ Sigma @ w * 252)
print("annualized return: %.1f%%" % (ann_ret * 100))
print("annualized vol:    %.1f%%" % (ann_vol * 100))
print("Sharpe (rf=0):     %.2f" % (ann_ret / ann_vol))

weights = pd.Series(w, index=tickers).sort_values(ascending=False)
print("\ntop longs:\n", weights.head(8).round(3))
print("\nshorts:\n", weights[weights < 0].round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(np.arange(len(weights)), weights.values)
ax.set_xticks(np.arange(len(weights)))
ax.set_xticklabels(weights.index, rotation=90, fontsize=6)
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("weight"); ax.set_title("Markowitz weights (sum = 1)")


## Notes

- **$\Sigma$ is invertible here because $T > N$** (501 days vs 45 stocks). For the full 1500-stock universe ($N > T$) the sample covariance is singular — you'd need shrinkage (Ledoit-Wolf) or a factor model before $\Sigma^{-1}$ exists.
- **Shorts are allowed** (no long-only constraint); the budget-constrained weights can be negative.
- **$\mu$ is noisy.** Sample mean returns are a notoriously bad estimator, so Markowitz weights are unstable — this is *the* known weakness, and the motivation for shrinkage / factor / Black-Litterman style fixes. Sweep $\gamma$ to trace the efficient frontier.
